In [6]:
import os

print(os.getcwd())

c:\Users\rekal\OneDrive\Desktop\MutualFundAnalytics\notebooks


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
funds = pd.read_csv(r'C:\Users\rekal\OneDrive\Desktop\MutualFundAnalytics\data\processed\01_fund_master.csv')
nav = pd.read_csv(r'C:\Users\rekal\OneDrive\Desktop\MutualFundAnalytics\data\processed\02_nav_history.csv')
txn = pd.read_csv(r'C:\Users\rekal\OneDrive\Desktop\MutualFundAnalytics\data\processed\08_investor_transactions.csv')
holdings = pd.read_csv(r'C:\Users\rekal\OneDrive\Desktop\MutualFundAnalytics\data\processed\09_portfolio_holdings.csv')

# Convert dates
nav['date'] = pd.to_datetime(nav['date'])
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

# Sort NAV history
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

print(nav.head())

   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [18]:
# Compute daily returns
nav['daily_return'] = (
    nav.groupby('amfi_code')['nav']
       .pct_change()
)

results = []

for code, df in nav.groupby('amfi_code'):

    r = df['daily_return'].dropna()

    if len(r) < 30:
        continue

    var95 = np.percentile(r, 5)
    cvar95 = r[r <= var95].mean()

    results.append({
        'amfi_code': code,
        'VaR_95': var95,
        'CVaR_95': cvar95,
        'observations': len(r)
    })

var_df = pd.DataFrame(results)

var_df = var_df.merge(
    funds[['amfi_code', 'scheme_name']],
    on='amfi_code',
    how='left'
)

# Save report
var_df.to_csv(
    'var_cvar_report.csv',
    index=False
)

# Display highest-risk funds
var_df.sort_values('VaR_95').head(10)

,amfi_code,VaR_95,CVaR_95,observations,scheme_name
22,119599,-0.026859,-0.032384,1149,SBI Small Cap Fund - Direct Plan - Growth
17,119095,-0.026188,-0.031667,1149,Axis Small Cap Fund - Regular - Growth
4,101207,-0.026021,-0.032459,1149,ABSL Small Cap Fund - Regular - Growth
11,118634,-0.025438,-0.032304,1149,Nippon India Small Cap Fund - Regular - Growth
21,119598,-0.024507,-0.030595,1149,SBI Small Cap Fund - Regular Plan - Growth
39,149324,-0.023483,-0.031036,1149,DSP Small Cap Fund - Regular - Growth
7,102886,-0.019220,-0.023251,1149,UTI Mid Cap Fund - Regular - Growth
2,100033,-0.019034,-0.023456,1149,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
25,120505,-0.018892,-0.024342,1149,ICICI Pru Midcap Fund - Regular - Growth
16,119094,-0.018480,-0.024260,1149,Axis Midcap Fund - Regular - Growth


In [19]:
first_txn = (
    txn.groupby('investor_id')['transaction_date']
       .min()
       .dt.year
       .rename('cohort_year')
)

txn = txn.merge(
    first_txn,
    on='investor_id'
)

cohort = (
    txn.groupby('cohort_year')
       .agg(
           avg_sip_amount=('amount_inr','mean'),
           total_invested=('amount_inr','sum'),
           investors=('investor_id','nunique')
       )
       .reset_index()
)

# Top fund preference
pref = (
    txn.groupby(['cohort_year','amfi_code'])
       .size()
       .reset_index(name='transactions')
)

idx = (
    pref.groupby('cohort_year')['transactions']
        .idxmax()
)

top_pref = pref.loc[idx]

top_pref = top_pref.merge(
    funds[['amfi_code','scheme_name']],
    on='amfi_code',
    how='left'
)

cohort = cohort.merge(
    top_pref[['cohort_year','scheme_name']],
    on='cohort_year',
    how='left'
)

cohort.rename(
    columns={'scheme_name':'top_fund_preference'},
    inplace=True
)

cohort.to_csv(
    'cohort_analysis.csv',
    index=False
)


In [20]:
sip = txn[
    txn['transaction_type']=='SIP'
].copy()

def sip_gap(df):

    df = df.sort_values('transaction_date')

    gaps = (
        df['transaction_date']
          .diff()
          .dt.days
          .dropna()
    )

    return pd.Series({
        'sip_count': len(df),
        'avg_gap': gaps.mean() if len(gaps)>0 else np.nan
    })

gap_df = (
    sip.groupby('investor_id')
       .apply(sip_gap)
       .reset_index()
)

gap_df = gap_df[
    gap_df['sip_count']>=6
]

gap_df['status'] = np.where(
    gap_df['avg_gap']>35,
    'at-risk',
    'healthy'
)

gap_df.to_csv(
    'sip_continuity.csv',
    index=False
)

continuity_rate = (
    (gap_df['status']=='healthy')
    .mean()
)

print(
    f'SIP continuity rate: {continuity_rate:.1%}'
)

gap_df.head()

SIP continuity rate: 2.2%


C:\Users\rekal\AppData\Local\Temp\ipykernel_8036\2245429543.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(sip_gap)


,investor_id,sip_count,avg_gap,status
3,INV000004,6.0,85.400000,at-risk
7,INV000008,6.0,70.400000,at-risk
9,INV000010,6.0,64.800000,at-risk
10,INV000011,7.0,40.166667,at-risk
11,INV000012,8.0,57.000000,at-risk


In [22]:
# Ensure daily returns exist
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

rolling = []

for code, df in nav.groupby('amfi_code'):

    df = df.sort_values('date').copy()
    r = df['daily_return']

    sharpe = (
        r.rolling(90).mean() /
        r.rolling(90).std()
    ) * np.sqrt(252)

    temp = pd.DataFrame({
        'date': df['date'],
        'amfi_code': code,
        'rolling_sharpe': sharpe
    })

    rolling.append(temp)

rolling_df = pd.concat(rolling, ignore_index=True)

print(rolling_df.head())
print('Rows:', len(rolling_df))

        date  amfi_code  rolling_sharpe
0 2022-01-03     100016             NaN
1 2022-01-04     100016             NaN
2 2022-01-05     100016             NaN
3 2022-01-06     100016             NaN
4 2022-01-07     100016             NaN
Rows: 46000


In [23]:
fund_sharpe = (
    rolling_df.groupby('amfi_code')['rolling_sharpe']
              .mean()
              .reset_index()
)

fund_sharpe = fund_sharpe.merge(
    funds[['amfi_code', 'scheme_name', 'risk_category']],
    on='amfi_code',
    how='left'
)

fund_sharpe.to_csv('fund_recommendations.csv', index=False)

fund_sharpe.head()

,amfi_code,rolling_sharpe,scheme_name,risk_category
0,100016,0.337507,HDFC Top 100 Fund - Regular Plan - Growth,Moderate
1,100025,1.125502,HDFC Short Term Debt Fund - Regular - Growth,Low
2,100033,1.586154,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,High
3,101206,1.484648,ABSL Frontline Equity Fund - Regular - Growth,Moderate
4,101207,0.416205,ABSL Small Cap Fund - Regular - Growth,Very High


In [24]:
fund_sharpe = (
    rolling_df.groupby('amfi_code')['rolling_sharpe']
              .mean()
              .reset_index()
)

fund_sharpe = fund_sharpe.merge(
    funds[['amfi_code', 'scheme_name', 'risk_category']],
    on='amfi_code',
    how='left'
)

fund_sharpe.to_csv('fund_recommendations.csv', index=False)

fund_sharpe.sort_values('rolling_sharpe', ascending=False).head(10)

,amfi_code,rolling_sharpe,scheme_name,risk_category
27,120507,13.558546,ICICI Pru Liquid Fund - Regular - Growth,Low
31,120844,12.478178,Kotak Liquid Fund - Regular - Growth,Low
5,101208,11.995642,ABSL Liquid Fund - Regular - Growth,Low
30,120843,1.997692,Kotak Flexicap Fund - Regular - Growth,Moderately High
34,148567,1.890422,Mirae Asset Large Cap Fund - Regular - Growth,Moderate
19,119551,1.727788,SBI Bluechip Fund - Regular Plan - Growth,Moderate
2,100033,1.586154,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,High
24,120504,1.561429,ICICI Pru Bluechip Fund - Direct - Growth,Moderate
38,149323,1.544997,DSP Midcap Fund - Regular - Growth,High
18,119120,1.529147,SBI Magnum Gilt Fund - Regular Plan - Growth,Low


In [25]:
summary = (
    fund_sharpe.merge(
        var_df[['amfi_code', 'VaR_95', 'CVaR_95']],
        on='amfi_code',
        how='left'
    )
)

summary = summary.sort_values(
    'rolling_sharpe',
    ascending=False
)

summary.to_csv(
    'advanced_fund_scorecard.csv',
    index=False
)

summary.head(10)

,amfi_code,rolling_sharpe,scheme_name,risk_category,VaR_95,CVaR_95
27,120507,13.558546,ICICI Pru Liquid Fund - Regular - Growth,Low,-0.000222,-0.000373
31,120844,12.478178,Kotak Liquid Fund - Regular - Growth,Low,-0.000285,-0.000411
5,101208,11.995642,ABSL Liquid Fund - Regular - Growth,Low,-0.000269,-0.000422
30,120843,1.997692,Kotak Flexicap Fund - Regular - Growth,Moderately High,-0.014508,-0.018381
34,148567,1.890422,Mirae Asset Large Cap Fund - Regular - Growth,Moderate,-0.013560,-0.017623
19,119551,1.727788,SBI Bluechip Fund - Regular Plan - Growth,Moderate,-0.012846,-0.016397
2,100033,1.586154,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,High,-0.019034,-0.023456
24,120504,1.561429,ICICI Pru Bluechip Fund - Direct - Growth,Moderate,-0.013728,-0.017494
38,149323,1.544997,DSP Midcap Fund - Regular - Growth,High,-0.017882,-0.022607
18,119120,1.529147,SBI Magnum Gilt Fund - Regular Plan - Growth,Low,-0.003938,-0.005014


In [26]:
import os

os.makedirs('outputs', exist_ok=True)

In [28]:
var_df.to_csv('outputs/var_cvar_report.csv', index=False)
cohort.to_csv('outputs/cohort_analysis.csv', index=False)
gap_df.to_csv('outputs/sip_continuity.csv', index=False)
fund_sharpe.to_csv('outputs/fund_recommendations.csv', index=False)


plt.savefig('outputs/rolling_sharpe_chart.png', dpi=300)

<Figure size 1200x600 with 0 Axes>

In [29]:
import pandas as pd

funds = pd.read_csv('outputs/fund_recommendations.csv')

risk = input('Risk Appetite (Low / Moderate / High): ').strip()

rec = (
    funds[
        funds['risk_category'].str.lower() == risk.lower()
    ]
    .sort_values('rolling_sharpe', ascending=False)
    .head(3)
)

print('\nTop 3 Recommended Funds\n')
print(
    rec[
        ['scheme_name', 'risk_category', 'rolling_sharpe']
    ].to_string(index=False)
)


Top 3 Recommended Funds

Empty DataFrame
Columns: [scheme_name, risk_category, rolling_sharpe]
Index: []
